# Upload de Datos a MinIO — EcoBici CABA

Este notebook sube los datasets de EcoBici al bucket `s3://data` de **MinIO**, que actúa como el _data lake_ local del stack Docker.

Se suben dos categorías de archivos:
- **Raw** → `ecobici_data.csv` (765 MB): el dataset crudo original. Es la fuente de verdad del pipeline; el DAG de ETL de Airflow lo leerá para producir los splits de entrenamiento.
- **Procesados** → `X_train`, `X_test`, `y_train`, `y_test`: los splits ya listos para entrenar, generados por el notebook de EDA. El DAG de Training los usa directamente.

---

**Prerequisito:** tener el stack Docker levantado antes de ejecutar este notebook.

```bash
cd Entrega_Operaciones/
docker compose --profile all up -d
```

MinIO es accesible desde la Mac en `http://localhost:9000` (puerto mapeado por Docker).
Desde dentro del stack Docker, los contenedores lo acceden en `http://s3:9000`.

---

## 0. Variables de entorno y configuración

Se exportan las credenciales de MinIO como variables de entorno. boto3 las toma automáticamente al crear el cliente, sin necesidad de pasarlas explícitamente en el código.

In [ ]:
%env AWS_ACCESS_KEY_ID=minio
%env AWS_SECRET_ACCESS_KEY=minio123
%env AWS_ENDPOINT_URL_S3=http://localhost:9000

In [ ]:
import boto3
from pathlib import Path

In [ ]:
# ── Bucket y rutas ───────────────────────────────────────────────────────────
DATA_BUCKET = "data"

BASE_DIR   = Path("../../EDA/dataset")
RAW_FILE   = BASE_DIR / "ecobici_data.csv"
PROC_FILES = {
    "X_train": BASE_DIR / "X_train.csv",
    "X_test":  BASE_DIR / "X_test.csv",
    "y_train": BASE_DIR / "y_train.csv",
    "y_test":  BASE_DIR / "y_test.csv",
}

# Estructura en MinIO:
#   s3://data/ecobici/raw/         → dataset crudo
#   s3://data/ecobici/processed/   → splits listos para entrenar
RAW_PREFIX  = "ecobici/raw"
PROC_PREFIX = "ecobici/processed"

---

## 1. Conexión a MinIO

Se crea el cliente de boto3 apuntando al endpoint local de MinIO. boto3 usa la API de S3 de AWS, que MinIO implementa de forma compatible.

Se listan los buckets disponibles para verificar que la conexión es correcta. Deben aparecer `data` y `mlflow` (creados automáticamente al levantar Docker).

In [ ]:
# ── Cliente S3/MinIO ─────────────────────────────────────────────────────────
# boto3 toma AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY y AWS_ENDPOINT_URL_S3
# directamente de las variables de entorno seteadas arriba
s3 = boto3.client("s3")

buckets = s3.list_buckets()
print("Buckets disponibles en MinIO:")
for b in buckets["Buckets"]:
    print(f"  · {b['Name']}")

---

## 2. Función auxiliar de upload

Se define una función reutilizable que sube un archivo local a MinIO e imprime el progreso. boto3 soporta un `Callback` que se invoca cada vez que se transfiere un chunk de bytes — útil para archivos grandes como el CSV crudo de 765 MB.

In [ ]:
class _ProgressPrinter:
    """Callback para boto3.upload_file que imprime el progreso del upload."""

    def __init__(self, filename: str, total_bytes: int):
        self._filename    = filename
        self._total_bytes = total_bytes
        self._uploaded    = 0

    def __call__(self, bytes_transferred: int):
        self._uploaded += bytes_transferred
        pct      = self._uploaded / self._total_bytes * 100
        mb_up    = self._uploaded    / 1_048_576
        mb_total = self._total_bytes / 1_048_576
        print(f"\r  {self._filename}: {pct:5.1f}%  ({mb_up:.0f} / {mb_total:.0f} MB)",
              end="", flush=True)
        if self._uploaded >= self._total_bytes:
            print("  ✓")


def upload_file(s3_client, local_path: Path, bucket: str, key: str):
    """Sube `local_path` a `s3://<bucket>/<key>` mostrando progreso."""
    if not local_path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {local_path}")

    size = local_path.stat().st_size
    print(f"Subiendo → s3://{bucket}/{key}")
    s3_client.upload_file(
        str(local_path),
        bucket,
        key,
        Callback=_ProgressPrinter(local_path.name, size),
    )

---

## 3. Upload del dataset crudo

Se sube `ecobici_data.csv` al prefijo `ecobici/raw/` dentro del bucket `data`.

Este archivo es el punto de entrada del pipeline completo. El **DAG de ETL** de Airflow lo leerá desde MinIO, aplicará la misma lógica de limpieza y feature engineering del notebook EDA, y escribirá los splits procesados de vuelta en MinIO.

In [ ]:
# ── Raw ──────────────────────────────────────────────────────────────────────
upload_file(s3, RAW_FILE, DATA_BUCKET, f"{RAW_PREFIX}/{RAW_FILE.name}")

---

## 4. Upload de los splits procesados

Se suben los cuatro archivos generados por el notebook de EDA al prefijo `ecobici/processed/`.

Tener los splits procesados en MinIO permite al **DAG de Training** arrancar directamente sin necesidad de correr el DAG de ETL primero — útil para pruebas o re-entrenamientos rápidos cuando los datos ya están listos.

In [ ]:
# ── Procesados ───────────────────────────────────────────────────────────────
for nombre, path in PROC_FILES.items():
    upload_file(s3, path, DATA_BUCKET, f"{PROC_PREFIX}/{path.name}")

---

## 5. Verificación

Se listan todos los objetos dentro del prefijo `ecobici/` para confirmar que el upload fue exitoso y que los tamaños son correctos.

In [ ]:
# ── Verificación ─────────────────────────────────────────────────────────────
print(f"Contenido de s3://{DATA_BUCKET}/ecobici/\n")
print(f"  {'Archivo':<55} {'Tamaño':>10}")
print(f"  {'-'*55} {'-'*10}")

paginator = s3.get_paginator("list_objects_v2")
total_mb  = 0

for page in paginator.paginate(Bucket=DATA_BUCKET, Prefix="ecobici/"):
    for obj in page.get("Contents", []):
        size_mb   = obj["Size"] / 1_048_576
        total_mb += size_mb
        print(f"  {obj['Key']:<55} {size_mb:>9.1f} MB")

print(f"  {'-'*55} {'-'*10}")
print(f"  {'Total':<55} {total_mb:>9.1f} MB")